In [1]:
import pandas as pd
import numpy as np

In [2]:
ENTREPOT_PATH = '/home/tbadie/Bureau/data/data_entrepot_outils/'
donnees = {}

def import_dfs(df_names, path_data, sep = ','):
    i = 0
    for df_name in df_names: 
        donnees[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, low_memory=False).replace({'\r\n': '\n'}, regex=True)

tables = [
    'sdc',
    'synthetise',
    'connection_synthetise',
    'noeuds_synthetise',
    'connection_realise',
    'noeuds_realise',
    'zone',
    'parcelle',
    'composant_culture',
    'espece',

    'entite_unique_par_sdc_nettoyage',

    'connection_synthetise_restructure',
    'noeuds_synthetise_restructure',

    'poids_connexions_synthetise_rotation',
    'poids_noeuds_realise',

    'typologie_culture_outils_dirodur',
    'typologie_can_culture',
    'date_de_semis_outils_dirodur'
    ]

# import des données
import_dfs(tables, ENTREPOT_PATH, sep = ',')

In [3]:
def initialisation_donnees(donnees):

    poids_S = donnees['poids_connexions_synthetise_rotation'][['connexion_id','poids_conx_agregation_norm_synth']].rename(columns={'connexion_id':'connection_synthetise_id'}).copy()
    poids_R = donnees['poids_noeuds_realise'][['noeuds_realise_id','poids_surface_developpee_normalisee']].copy()
    date_semis = donnees['date_de_semis_outils_dirodur'][['culture_id','saison_semis_detect_via_intv']].copy()

    sdc = donnees['sdc'][['id','filiere']].rename(columns={'id':'sdc_id'}).copy()

    unique_sdc = donnees['entite_unique_par_sdc_nettoyage'].copy()
    sdc_real = sdc.loc[sdc['sdc_id'].isin(unique_sdc.loc[unique_sdc['entite_retenue'] == 'realise_retenu','sdc_id'])]
    synthetise = donnees['synthetise'][['id','sdc_id']].rename(columns={'id':'synthetise_id'}).copy()
    synthetise = synthetise.loc[synthetise['synthetise_id'].isin(unique_sdc['entite_retenue'].unique())]

    cnx_s = donnees['connection_synthetise'][['id','cible_noeuds_synthetise_id']].rename(columns={'id':'connection_synthetise_id', 'cible_noeuds_synthetise_id':'noeuds_synthetise_id'}).copy()
    cnx_s_rest = donnees['connection_synthetise_restructure'].rename(columns={'id':'connection_synthetise_id'}).copy()
    nd_s = donnees['noeuds_synthetise'][['id','synthetise_id']].rename(columns={'id':'noeuds_synthetise_id'}).copy()
    nd_s_rest = donnees['noeuds_synthetise_restructure'].rename(columns={'id':'noeuds_synthetise_id'}).copy()

    cnx_r = donnees['connection_realise'][['id','cible_noeuds_realise_id','culture_intermediaire_id']].rename(columns={'id':'connexion_realise_id','cible_noeuds_realise_id':'noeuds_realise_id'})
    nd_r = donnees['noeuds_realise'].rename(columns={'id':'noeuds_realise_id'}).copy()
    zone = donnees['zone'][['id','parcelle_id']].rename(columns={'id':'zone_id'}).copy()
    parcelle = donnees['parcelle'][['id','sdc_id']].rename(columns={'id':'parcelle_id'}).copy()

    cropsp = donnees['composant_culture'][['id','espece_id','culture_id']].rename(columns={'id':'composant_culture_id'}).copy()
    sp = donnees['espece'][['id','libelle_espece_botanique','typodirodur_espece','typodirodur_espece_precise','typodirodur_espece_famille_bota','typodirodur_espece_periode_semis']].rename(columns={'id':'espece_id'}).copy()
    typo_dirodur = donnees['typologie_culture_outils_dirodur'][['culture_id', 'typodirodur_culture', 'culture_est_avec_compagne', 
                                                                'culture_est_annuelle_asso', 'culture_est_prairie', 'typo_cpg']].copy()
    typo_can = donnees['typologie_can_culture'][['culture_id','typocan_culture_sans_compagne']].copy()

    sp = cropsp.merge(sp, how = 'left', on = 'espece_id')

    sp['ponderation_composant'] = 1/sp.groupby('culture_id')['composant_culture_id'].transform("count")

    # merge outer pour les noeud sur les connexion en réalisé car tous les noeuds n'ont pas forcément de connexion
    # merge inner avec synthetise et sdc pour n'avoir que les entite unique par sdc !
    itk_s = cnx_s.merge(cnx_s_rest, how='left', on='connection_synthetise_id').merge(nd_s, how='left', on='noeuds_synthetise_id').merge(nd_s_rest, how='left', on='noeuds_synthetise_id').merge(synthetise, how='inner', on='synthetise_id')
    itk_r = cnx_r.merge(nd_r, how='outer', on ='noeuds_realise_id').merge(zone, how='left', on='zone_id').merge(parcelle, how='left', on='parcelle_id').merge(sdc_real, how='inner', on='sdc_id')
    itk = pd.concat([itk_s, itk_r])

    itk = itk[['connection_synthetise_id', 'noeuds_realise_id', 'culture_id', 'culture_intermediaire_id', 'synthetise_id', 'sdc_id']]
    composant_itk = itk.merge(sp, on='culture_id', how='left').merge(typo_dirodur, how = 'left', on = 'culture_id').merge(typo_can, how = 'left', on = 'culture_id')
    composant_itk = composant_itk.merge(poids_S, how='left', on='connection_synthetise_id')
    composant_itk = composant_itk.merge(poids_R, how='left', on='noeuds_realise_id')

    composant_itk = composant_itk.merge(date_semis, how='left', on='culture_id')
    composant_itk['saison_semis_detect_via_intv'] = composant_itk['typodirodur_espece_periode_semis'].fillna(composant_itk['saison_semis_detect_via_intv'])
    composant_itk.drop(columns = 'saison_semis_detect_via_intv', inplace=True)

    # On calcule les poids par composant au seins du sdc (ou synthetise). On prends le poids de connexion ou le poids de noeuds selon la méthode de saisie (R ou S)
    composant_itk['poids_composant_dans_sdc'] = np.where(composant_itk['connection_synthetise_id'].notna(),
                                                        composant_itk['ponderation_composant'] * composant_itk['poids_conx_agregation_norm_synth'],
                                                        composant_itk['ponderation_composant'] * composant_itk['poids_surface_developpee_normalisee'])

    return composant_itk

df = initialisation_donnees(donnees)

In [ ]:
### MISE EN PLACE DES FONCTION CALCULANT LES INDICATEURS ###

def richness(p):
    return len(p.index)

def shannon(p):
    sh = -(p * np.log2(p)).sum()
    if sh == -0:
        return 0
    return sh

def evenness(p):
    s = len(p)
    if s <= 1:
        return np.nan
    return shannon(p) / np.log2(s)

def simpson(p):
    return (p**2).sum()

def inverse_simpson(p):
    s = simpson(p)
    if pd.isna(s) or s == 0:
        return np.nan
    return 1 / s

list_typo_can = [
            'Céréales à paille hiver',
            'Céréales à paille printemps',
            'Mélange fourrager',
            'Légume',
            'Protéagineux',
            'Maïs',
            'Prairie temporaire',
            'Colza',
            'Tournesol',
            'Oléagineux (hors Colza et Tournesol)',
            'Pomme de terre',
            'Lin',
            'Betterave',
            'NoInput-sp'
        ]

def compute_typology_metrics(df, typology_col, prefix, cols_needed_for_proportion=None):
    # Il a certaines cultures en absentes (==> poids = NaN) comme souvent pour les Précédents fictifs par exemple
    df = df[df["poids_composant_dans_sdc"].notna()]

    # On ajoute la modalité Inconnu pour ne pas sous ou sur estimé les proportions des autres modalités (groupby excluant par défaut les NaN dasn la typology_col)
    df.loc[:,typology_col] = df[typology_col].fillna("Inconnu")
    proportions = df.groupby(typology_col)["poids_composant_dans_sdc"].sum()

    # Il y a potentiellement des modalité avec une somme de proportion à 0%, on les retire
    proportions =  proportions[proportions > 0]

    # Le sdc n'a pas les poids associés à chaque culture ou n'avait que des poids à 0% ou que des Nan
    if proportions.empty and typology_col in ['typocan_culture_sans_compagne', 'typodirodur_culture'] :       
        return pd.Series({
            f"{prefix}_richesse": int(0),
            f"{prefix}_shannon": np.nan,
            f"{prefix}_evenness": np.nan,
            f"{prefix}_simpson": np.nan,
            f"{prefix}_inverse_simpson": np.nan,
        })
    elif proportions.empty and typology_col not in ['typocan_culture_sans_compagne', 'typodirodur_culture']  :       
        return pd.Series({
            f"{prefix}_richesse": int(0),
            f"{prefix}_shannon": np.nan,
        })
    
    # Calculs des indicateurs
    proportions = proportions / proportions.sum()

    if typology_col in ['typocan_culture_sans_compagne', 'typodirodur_culture'] :
        metrics = {
            f"{prefix}_richesse": int(richness(proportions)),
            f"{prefix}_shannon": shannon(proportions),
            f"{prefix}_evenness": evenness(proportions),
            f"{prefix}_simpson": simpson(proportions),
            f"{prefix}_inverse_simpson": inverse_simpson(proportions),
            f"{prefix}_proportion_max": max(proportions),
        }
    else : 
        metrics = {
            f"{prefix}_richesse": int(richness(proportions)),
            f"{prefix}_shannon": shannon(proportions),
        }

    # Calculs des proportions
    # Cas des famille botanique, on combine la proportion de toutes les autres familles qu les 3 principales
    if typology_col == 'typodirodur_espece_famille_bota' :
        mask = proportions.index.isin(["Poaceae", "Fabaceae", "Brassicaceae"])
        others = proportions[~mask].sum()
        proportions = proportions[mask].copy()
        proportions["Autres_familles"] = others

    if typology_col == 'typocan_culture_sans_compagne' :
        mask = proportions.index.isin(list_typo_can)
        others = proportions[~mask].sum()
        proportions = proportions[mask].copy()
        proportions["Autres_cultures_can"] = others

    prefix_proportion = 'prop'
    if typology_col == 'typocan_culture_sans_compagne' :
        prefix_proportion = 'prop_surface_can'

    if cols_needed_for_proportion is not None:
        for category in cols_needed_for_proportion:
            metrics[f"{prefix_proportion}_{category.lower().replace('é', 'e').replace(' ', '_')}"] = proportions.get(category, 0)

    return metrics


### UTILISATION DES FONCTION D'INDICATEURS ###

result = (
    df.groupby(["sdc_id"])
    .apply(
        lambda sdc: pd.DataFrame([
            {
                'synthetise_id': sdc['synthetise_id'].iloc[0] if any(sdc['synthetise_id'].notna()) else None,
                **compute_typology_metrics(sdc, "typodirodur_culture", "typodirodur_culture"),
                **compute_typology_metrics(sdc, "typodirodur_espece", "typodirodur_espece"),
                **compute_typology_metrics(sdc, "libelle_espece_botanique", "espece_bota"),
                **compute_typology_metrics(sdc, "typodirodur_espece_famille_bota", "famille_bota", ["Poaceae", "Fabaceae", "Brassicaceae", 'Autres_familles']),
                **compute_typology_metrics(sdc, "typodirodur_espece_periode_semis", "saison_semis", ["printemps", "ete", "automne", 'hiver', 'pluriannuel']),
                "prop_culture_avec_compagne": sdc.loc[sdc["culture_est_avec_compagne"] == "oui", "poids_composant_dans_sdc"].sum(),
                "prop_association": sdc.loc[sdc["culture_est_annuelle_asso"] == "oui", "poids_composant_dans_sdc"].sum(),
                "prop_prairie": sdc.loc[sdc["culture_est_prairie"] == "oui", "poids_composant_dans_sdc"].sum(),
                "prop_culture_intermediaire": sdc.loc[sdc["culture_intermediaire_id"].notna(), "poids_composant_dans_sdc"].sum(),
                "prop_culture_porte_graine": sdc.loc[sdc["typo_cpg"].notna(), "poids_composant_dans_sdc"].sum(),
                # pour la CAN (pas dispo dans la doc datagrosyst)
                **compute_typology_metrics(sdc, "typocan_culture_sans_compagne", "typocan_culture", (list_typo_can+['Autres_cultures_can'])),
            }
        ]),
        include_groups=False,
    )
    .reset_index()
).drop(columns='level_1')

for col in [col for col in result.columns if 'richesse' in col.lower()]:
    result[col] = result[col].astype('Int64')
    
result = result[[
    # Index
    'sdc_id',
    'synthetise_id',
    # Typo culture
    'typodirodur_culture_richesse',
    'typodirodur_culture_shannon',
    'typodirodur_culture_evenness',
    'typodirodur_culture_simpson',
    'typodirodur_culture_inverse_simpson',
    'typodirodur_culture_proportion_max',
    'prop_association',
    'prop_culture_avec_compagne',
    'prop_prairie',
    'prop_culture_intermediaire',
    'prop_culture_porte_graine',
    # Typo espece
    'typodirodur_espece_richesse',
    'typodirodur_espece_shannon',
    # Espece bota
    'espece_bota_richesse',
    'espece_bota_shannon',
    # Famille bota
    'famille_bota_richesse',
    'famille_bota_shannon',
    'prop_poaceae',
    'prop_fabaceae',
    'prop_brassicaceae',
    'prop_autres_familles',
    # Saison semis
    'saison_semis_richesse',
    'saison_semis_shannon',
    'prop_printemps',
    'prop_ete',
    'prop_automne',
    'prop_hiver',
    # typologie CAN
    'typocan_culture_richesse',
    'typocan_culture_shannon',
    'typocan_culture_evenness',
    'typocan_culture_simpson',
    'typocan_culture_inverse_simpson',
    # Proportion CAN
    'prop_surface_can_cereales_à_paille_hiver',
    'prop_surface_can_cereales_à_paille_printemps',
    'prop_surface_can_maïs',
    'prop_surface_can_colza',
    'prop_surface_can_tournesol',
    'prop_surface_can_oleagineux_(hors_colza_et_tournesol)',
    'prop_surface_can_proteagineux',
    'prop_surface_can_melange_fourrager',
    'prop_surface_can_lin',
    'prop_surface_can_pomme_de_terre',
    'prop_surface_can_betterave',
    'prop_surface_can_legume',
    'prop_surface_can_prairie_temporaire',
    'prop_surface_can_autres_cultures_can'
    ]]

In [5]:
result

,sdc_id,synthetise_id,typodirodur_culture_richesse,typodirodur_culture_shannon,typodirodur_culture_evenness,typodirodur_culture_simpson,typodirodur_culture_inverse_simpson,typodirodur_culture_proportion_max,prop_association,prop_culture_avec_compagne,...,prop_surface_can_tournesol,prop_surface_can_oleagineux_(hors_colza_et_tournesol),prop_surface_can_proteagineux,prop_surface_can_melange_fourrager,prop_surface_can_lin,prop_surface_can_pomme_de_terre,prop_surface_can_betterave,prop_surface_can_legume,prop_surface_can_prairie_temporaire,prop_surface_can_autres_cultures_can
0,fr.inra.agrosyst.api.entities.GrowingSystem_00...,None,4,1.786799,0.893399,0.333443,2.999009,0.496144,0.0,0.0,...,0.000000,0.0,0.128535,0.00,0.0,0.0,0.0,0.0,0.496144,0.00
1,fr.inra.agrosyst.api.entities.GrowingSystem_00...,None,1,0.000000,NaN,1.000000,1.000000,1.000000,0.0,0.0,...,0.000000,0.0,0.000000,0.00,0.0,0.0,0.0,1.0,0.000000,0.00
2,fr.inra.agrosyst.api.entities.GrowingSystem_00...,fr.inra.agrosyst.api.entities.practiced.Practi...,2,1.000000,1.000000,0.500000,2.000000,0.500000,0.0,0.0,...,0.000000,0.0,0.000000,0.00,0.0,0.0,0.0,0.0,0.000000,0.00
3,fr.inra.agrosyst.api.entities.GrowingSystem_00...,fr.inra.agrosyst.api.entities.practiced.Practi...,6,2.195462,0.849321,0.275000,3.636364,0.450000,0.2,0.0,...,0.000000,0.0,0.000000,0.15,0.0,0.0,0.0,0.0,0.450000,0.15
4,fr.inra.agrosyst.api.entities.GrowingSystem_00...,fr.inra.agrosyst.api.entities.practiced.Practi...,3,1.475336,0.930833,0.383450,2.607902,0.500000,0.0,0.0,...,0.000000,0.0,0.000000,0.00,0.0,0.0,0.0,0.0,0.000000,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22212,fr.inra.agrosyst.api.entities.GrowingSystem_ff...,None,5,2.281073,0.982405,0.212375,4.708651,0.297001,0.0,0.0,...,0.192954,0.0,0.000000,0.00,0.0,0.0,0.0,0.0,0.000000,0.00
22213,fr.inra.agrosyst.api.entities.GrowingSystem_ff...,fr.inra.agrosyst.api.entities.practiced.Practi...,5,2.155639,0.928383,0.250000,4.000000,0.375000,0.0,0.0,...,0.000000,0.0,0.000000,0.00,0.0,0.0,0.0,0.0,0.500000,0.00
22214,fr.inra.agrosyst.api.entities.GrowingSystem_ff...,None,7,2.351843,0.837744,0.235425,4.247629,0.367946,0.0,0.0,...,0.000000,0.0,0.038767,0.00,0.0,0.0,0.0,0.0,0.108874,0.00
22215,fr.inra.agrosyst.api.entities.GrowingSystem_ff...,None,2,0.918296,0.918296,0.555556,1.800000,0.666667,0.0,0.0,...,0.000000,0.0,0.000000,0.00,0.0,0.0,0.0,1.0,0.000000,0.00


In [ ]:
cols_CAN = [
'typocan_culture_richesse',
'typocan_culture_shannon',
'typocan_culture_evenness',
'typocan_culture_simpson',
'typocan_culture_inverse_simpson',

'prop_surface_can_cereales_à_paille_hiver',
'prop_surface_can_cereales_à_paille_printemps',
'prop_surface_can_maïs',
'prop_surface_can_colza',
'prop_surface_can_tournesol',
'prop_surface_can_oleagineux_(hors_colza_et_tournesol)',
'prop_surface_can_proteagineux',
'prop_surface_can_melange_fourrager',
'prop_surface_can_lin',
'prop_surface_can_pomme_de_terre',
'prop_surface_can_betterave',
'prop_surface_can_legume',
'prop_surface_can_prairie_temporaire',
'prop_surface_can_autres_cultures_can'
]

In [7]:
list_sdc_id = list(result.sample(50)['sdc_id'])

In [8]:
TEST_PATH = '/home/tbadie/Bureau/catalogue_script_agrosyst/02_outils/tests/data/test_get_indicateur_diversite_outils_dirodur/'

# Créer les nouvelles données filtrées
sdc2 = donnees['sdc'].loc[donnees['sdc']['id'].isin(list_sdc_id)]

synthetise2 = donnees['synthetise'].loc[donnees['synthetise']['sdc_id'].isin(sdc2['id'])]
ndS2 = donnees['noeuds_synthetise'].loc[donnees['noeuds_synthetise']['synthetise_id'].isin(synthetise2['id'])]
cxS2 = donnees['connection_synthetise'].loc[(donnees['connection_synthetise']['cible_noeuds_synthetise_id'].isin(ndS2['id'])) |  
                                            (donnees['connection_synthetise']['source_noeuds_synthetise_id'].isin(ndS2['id']))   ]

parcelle2 = donnees['parcelle'].loc[donnees['parcelle']['sdc_id'].isin(sdc2['id'])]
zone2 = donnees['zone'].loc[donnees['zone']['parcelle_id'].isin(parcelle2['id'])]
ndR2 = donnees['noeuds_realise'].loc[donnees['noeuds_realise']['zone_id'].isin(zone2['id'])]
cxR2 = donnees['connection_realise'].loc[(donnees['connection_realise']['cible_noeuds_realise_id'].isin(ndR2['id'])) |  
                                         (donnees['connection_realise']['source_noeuds_realise_id'].isin(ndR2['id']))   ]

ent_unique = donnees['entite_unique_par_sdc_nettoyage'].loc[donnees['entite_unique_par_sdc_nettoyage']['sdc_id'].isin(sdc2['id'])]
cxS2_rest = donnees['connection_synthetise_restructure'].loc[donnees['connection_synthetise_restructure']['id'].isin(cxS2['id'])]
ndS2_rest = donnees['noeuds_synthetise_restructure'].loc[donnees['noeuds_synthetise_restructure']['id'].isin(ndS2['id'])]
poids_cxS2 = donnees['poids_connexions_synthetise_rotation'].loc[donnees['poids_connexions_synthetise_rotation']['connexion_id'].isin(cxS2['id'])]
poids_ndR2 = donnees['poids_noeuds_realise'].loc[donnees['poids_noeuds_realise']['noeuds_realise_id'].isin(ndR2['id'])]

culture_id2 = set(ndS2_rest['culture_id'].tolist() + ndR2['culture_id'].tolist())

typo_diro = donnees['typologie_culture_outils_dirodur'].loc[donnees['typologie_culture_outils_dirodur']['culture_id'].isin(culture_id2)]
typo_can = donnees['typologie_can_culture'].loc[donnees['typologie_can_culture']['culture_id'].isin(culture_id2)]
date_semis = donnees['date_de_semis_outils_dirodur'].loc[donnees['date_de_semis_outils_dirodur']['culture_id'].isin(culture_id2)]
cc2 = donnees['composant_culture'].loc[donnees['composant_culture']['culture_id'].isin(culture_id2)]
esp2 = donnees['espece'].loc[donnees['espece']['id'].isin(cc2['espece_id'])]


# imprimer les nouvelles données filtrées
sdc2.to_csv(TEST_PATH + 'sdc.csv', index=False, sep=',')

synthetise2.to_csv(TEST_PATH + 'synthetise.csv', index=False, sep=',')
ndS2.to_csv(TEST_PATH + 'noeuds_synthetise.csv', index=False, sep=',')
cxS2.to_csv(TEST_PATH + 'connection_synthetise.csv', index=False, sep=',')

parcelle2.to_csv(TEST_PATH + 'parcelle.csv', index=False, sep=',')
zone2.to_csv(TEST_PATH + 'zone.csv', index=False, sep=',')
ndR2.to_csv(TEST_PATH + 'noeuds_realise.csv', index=False, sep=',')
cxR2.to_csv(TEST_PATH + 'connection_realise.csv', index=False, sep=',')

ent_unique.to_csv(TEST_PATH + 'entite_unique_par_sdc_nettoyage.csv', index=False, sep=',')
cxS2_rest.to_csv(TEST_PATH + 'connection_synthetise_restructure.csv', index=False, sep=',')
ndS2_rest.to_csv(TEST_PATH + 'noeuds_synthetise_restructure.csv', index=False, sep=',')
poids_cxS2.to_csv(TEST_PATH + 'poids_connexions_synthetise_rotation.csv', index=False, sep=',')
poids_ndR2.to_csv(TEST_PATH + 'poids_noeuds_realise.csv', index=False, sep=',')

typo_diro.to_csv(TEST_PATH + 'typologie_culture_outils_dirodur.csv', index=False, sep=',')
typo_can.to_csv(TEST_PATH + 'typologie_can_culture.csv', index=False, sep=',')
date_semis.to_csv(TEST_PATH + 'date_de_semis_outils_dirodur.csv', index=False, sep=',')
cc2.to_csv(TEST_PATH + 'composant_culture.csv', index=False, sep=',')
esp2.to_csv(TEST_PATH + 'espece.csv', index=False, sep=',')

In [9]:
del donnees
del result
donnees = {}
import_dfs(tables, TEST_PATH, sep = ',')

In [ ]:
df = initialisation_donnees(donnees)

### MISE EN PLACE DES FONCTION CALCULANT LES INDICATEURS ###

def richness(p):
    return len(p.index)

def shannon(p):
    sh = -(p * np.log2(p)).sum()
    if sh == -0:
        return 0
    return sh

def evenness(p):
    s = len(p)
    if s <= 1:
        return np.nan
    return shannon(p) / np.log2(s)

def simpson(p):
    return (p**2).sum()

def inverse_simpson(p):
    s = simpson(p)
    if pd.isna(s) or s == 0:
        return np.nan
    return 1 / s

list_typo_can = [
            'Céréales à paille hiver',
            'Céréales à paille printemps',
            'Mélange fourrager',
            'Légume',
            'Protéagineux',
            'Maïs',
            'Prairie temporaire',
            'Colza',
            'Tournesol',
            'Oléagineux (hors Colza et Tournesol)',
            'Pomme de terre',
            'Lin',
            'Betterave',
            'NoInput-sp'
        ]

def compute_typology_metrics(df, typology_col, prefix, cols_needed_for_proportion=None):
    # Il a certaines cultures en absentes (==> poids = NaN) comme souvent pour les Précédents fictifs par exemple
    df = df[df["poids_composant_dans_sdc"].notna()]

    # On ajoute la modalité Inconnu pour ne pas sous ou sur estimé les proportions des autres modalités (groupby excluant par défaut les NaN dasn la typology_col)
    df.loc[:,typology_col] = df[typology_col].fillna("Inconnu")
    proportions = df.groupby(typology_col)["poids_composant_dans_sdc"].sum()

    # Il y a potentiellement des modalité avec une somme de proportion à 0%, on les retire
    proportions =  proportions[proportions > 0]

    # Le sdc n'a pas les poids associés à chaque culture ou n'avait que des poids à 0% ou que des Nan
    if proportions.empty and typology_col in ['typocan_culture_sans_compagne', 'typodirodur_culture'] :       
        return pd.Series({
            f"{prefix}_richesse": int(0),
            f"{prefix}_shannon": np.nan,
            f"{prefix}_evenness": np.nan,
            f"{prefix}_simpson": np.nan,
            f"{prefix}_inverse_simpson": np.nan,
        })
    elif proportions.empty and typology_col not in ['typocan_culture_sans_compagne', 'typodirodur_culture']  :       
        return pd.Series({
            f"{prefix}_richesse": int(0),
            f"{prefix}_shannon": np.nan,
        })
    
    # Calculs des indicateurs
    proportions = proportions / proportions.sum()

    if typology_col in ['typocan_culture_sans_compagne', 'typodirodur_culture'] :
        metrics = {
            f"{prefix}_richesse": int(richness(proportions)),
            f"{prefix}_shannon": shannon(proportions),
            f"{prefix}_evenness": evenness(proportions),
            f"{prefix}_simpson": simpson(proportions),
            f"{prefix}_inverse_simpson": inverse_simpson(proportions),
            f"{prefix}_proportion_max": max(proportions),
        }
    else : 
        metrics = {
            f"{prefix}_richesse": int(richness(proportions)),
            f"{prefix}_shannon": shannon(proportions),
        }

    # Calculs des proportions
    # Cas des famille botanique, on combine la proportion de toutes les autres familles qu les 3 principales
    if typology_col == 'typodirodur_espece_famille_bota' :
        mask = proportions.index.isin(["Poaceae", "Fabaceae", "Brassicaceae"])
        others = proportions[~mask].sum()
        proportions = proportions[mask].copy()
        proportions["Autres_familles"] = others

    if typology_col == 'typocan_culture_sans_compagne' :
        mask = proportions.index.isin(list_typo_can)
        others = proportions[~mask].sum()
        proportions = proportions[mask].copy()
        proportions["Autres_cultures_can"] = others

    prefix_proportion = 'prop'
    if typology_col == 'typocan_culture_sans_compagne' :
        prefix_proportion = 'prop_surface_can'

    if cols_needed_for_proportion is not None:
        for category in cols_needed_for_proportion:
            metrics[f"{prefix_proportion}_{category.lower().replace('é', 'e').replace(' ', '_')}"] = proportions.get(category, 0)

    return metrics


### UTILISATION DES FONCTION D'INDICATEURS ###

result = (
    df.groupby(["sdc_id"])
    .apply(
        lambda sdc: pd.DataFrame([
            {
                'synthetise_id': sdc['synthetise_id'].iloc[0] if any(sdc['synthetise_id'].notna()) else None,
                **compute_typology_metrics(sdc, "typodirodur_culture", "typodirodur_culture"),
                **compute_typology_metrics(sdc, "typodirodur_espece", "typodirodur_espece"),
                **compute_typology_metrics(sdc, "libelle_espece_botanique", "espece_bota"),
                **compute_typology_metrics(sdc, "typodirodur_espece_famille_bota", "famille_bota", ["Poaceae", "Fabaceae", "Brassicaceae", 'Autres_familles']),
                **compute_typology_metrics(sdc, "typodirodur_espece_periode_semis", "saison_semis", ["printemps", "ete", "automne", 'hiver', 'pluriannuel']),
                "prop_culture_avec_compagne": sdc.loc[sdc["culture_est_avec_compagne"] == "oui", "poids_composant_dans_sdc"].sum(),
                "prop_association": sdc.loc[sdc["culture_est_annuelle_asso"] == "oui", "poids_composant_dans_sdc"].sum(),
                "prop_prairie": sdc.loc[sdc["culture_est_prairie"] == "oui", "poids_composant_dans_sdc"].sum(),
                "prop_culture_intermediaire": sdc.loc[sdc["culture_intermediaire_id"].notna(), "poids_composant_dans_sdc"].sum(),
                "prop_culture_porte_graine": sdc.loc[sdc["typo_cpg"].notna(), "poids_composant_dans_sdc"].sum(),
                # pour la CAN (pas dispo dans la doc datagrosyst)
                **compute_typology_metrics(sdc, "typocan_culture_sans_compagne", "typocan_culture", (list_typo_can+['Autres_cultures_can'])),
            }
        ]),
        include_groups=False,
    )
    .reset_index()
).drop(columns='level_1')

for col in [col for col in result.columns if 'richesse' in col.lower()]:
    result[col] = result[col].astype('Int64')
    
result = result[[
    # Index
    'sdc_id',
    'synthetise_id',
    # Typo culture
    'typodirodur_culture_richesse',
    'typodirodur_culture_shannon',
    'typodirodur_culture_evenness',
    'typodirodur_culture_simpson',
    'typodirodur_culture_inverse_simpson',
    'typodirodur_culture_proportion_max',
    'prop_association',
    'prop_culture_avec_compagne',
    'prop_prairie',
    'prop_culture_intermediaire',
    'prop_culture_porte_graine',
    # Typo espece
    'typodirodur_espece_richesse',
    'typodirodur_espece_shannon',
    # Espece bota
    'espece_bota_richesse',
    'espece_bota_shannon',
    # Famille bota
    'famille_bota_richesse',
    'famille_bota_shannon',
    'prop_poaceae',
    'prop_fabaceae',
    'prop_brassicaceae',
    'prop_autres_familles',
    # Saison semis
    'saison_semis_richesse',
    'saison_semis_shannon',
    'prop_printemps',
    'prop_ete',
    'prop_automne',
    'prop_hiver',
    # typologie CAN
    'typocan_culture_richesse',
    'typocan_culture_shannon',
    'typocan_culture_evenness',
    'typocan_culture_simpson',
    'typocan_culture_inverse_simpson',
    # Proportion CAN
    'prop_surface_can_cereales_à_paille_hiver',
    'prop_surface_can_cereales_à_paille_printemps',
    'prop_surface_can_maïs',
    'prop_surface_can_colza',
    'prop_surface_can_tournesol',
    'prop_surface_can_oleagineux_(hors_colza_et_tournesol)',
    'prop_surface_can_proteagineux',
    'prop_surface_can_melange_fourrager',
    'prop_surface_can_lin',
    'prop_surface_can_pomme_de_terre',
    'prop_surface_can_betterave',
    'prop_surface_can_legume',
    'prop_surface_can_prairie_temporaire',
    'prop_surface_can_autres_cultures_can'
    ]]

In [ ]:
def creer_df_tests(df, test_id, nb_par_colonne):
    lignes = []

    for colonne, n in nb_par_colonne.items():

        if colonne not in df.columns:
            raise ValueError(f"La colonne '{colonne}' n'existe pas.")

        serie = df[colonne]

        if n > len(serie):
            raise ValueError(
                f"Impossible de tirer {n} valeurs sans remise dans la colonne '{colonne}' "
                f"(seulement {len(serie)} valeurs disponibles)."
            )

        echantillon = serie.sample(n=n, replace=False)

        for idx, valeur in echantillon.items():
            lignes.append({
                "test_id": test_id,
                "index": idx,
                "valeur": valeur,
                "resultat": None,   # colonne vide
                "colonne": colonne
            })

    return pd.DataFrame(lignes)

nb_par_colonne = {
            'synthetise_id':4,
            # Typo culture
            'typodirodur_culture_richesse':15,
            'typodirodur_culture_shannon':15,
            'typodirodur_culture_evenness':15,
            'typodirodur_culture_simpson':15,
            'typodirodur_culture_inverse_simpson':15,
            'typodirodur_culture_proportion_max':15,
            'prop_association':30,
            'prop_culture_avec_compagne':30,
            'prop_prairie':30,
            'prop_culture_intermediaire':30,
            'prop_culture_porte_graine':50,
            # Typo espece
            'typodirodur_espece_richesse':15,
            'typodirodur_espece_shannon':15,
            # Espece bota
            'espece_bota_richesse':15,
            'espece_bota_shannon':15,
            # Famille bota
            'famille_bota_richesse':15,
            'famille_bota_shannon':15,
            'prop_poaceae':30,
            'prop_fabaceae':30,
            'prop_brassicaceae':30,
            'prop_autres_familles':30,
            # Saison semis
            'saison_semis_richesse':15,
            'saison_semis_shannon':15,
            'prop_printemps':30,
            'prop_ete':30,
            'prop_automne':30,
            'prop_hiver':30,
            # typologie CAN
            'typocan_culture_richesse':15,
            'typocan_culture_shannon':15,
            'typocan_culture_evenness':15,
            'typocan_culture_simpson':15,
            'typocan_culture_inverse_simpson':15,
            # Proportion CAN
            'prop_surface_can_cereales_à_paille_hiver':30,
            'prop_surface_can_cereales_à_paille_printemps':30,
            'prop_surface_can_maïs':30,
            'prop_surface_can_colza':30,
            'prop_surface_can_tournesol':30,
            'prop_surface_can_oleagineux_(hors_colza_et_tournesol)':30,
            'prop_surface_can_proteagineux':30,
            'prop_surface_can_melange_fourrager':30,
            'prop_surface_can_lin':30,
            'prop_surface_can_pomme_de_terre':30,
            'prop_surface_can_betterave':30,
            'prop_surface_can_legume':30,
            'prop_surface_can_prairie_temporaire':30,
            'prop_surface_can_autres_cultures_can':30
        }

result.set_index('sdc_id', inplace=True)

final_TU = creer_df_tests(result,
                          'test_get_indicateur_diversite_outils_dirodur', 
                          nb_par_colonne)

final_TU.to_csv('/home/tbadie/Bureau/TU_a_utiliser.csv', index=False)